# NovaBank Digital Production Monitoring Patterns

This notebook runs the v0.8 production-monitoring flow for **NovaBank Digital** (digital-banking track). It is the digital-banking counterpart of the Alpine Crest monitoring notebook and consumes the same v0.8 monitoring builders — batch scoring, alert decisions, reviewer/audit, alert-queue inspection, operational metrics, score/feature drift, and monitoring data-quality — on top of the v0.4 digital-banking supervised baseline and the v0.7 threshold recommender / feature explanations.

Learning goal: trace one digital Detection pattern (`digital_scam_to_mule`) from a fitted score, through threshold selection, into the production score / threshold / alert_decision / reviewer_action / audit tables, into an inspectable Alert queue and an operational summary, and out to drift and data-quality checks. The notebook keeps the limitation-aware framing visible — monitoring is a review prompt, not a verdict, and **a model should not be judged by headline accuracy**.

> Educational exercise on synthetic data. No real Client, account, or transaction data; no production-readiness, certification, or legal-advice claim. The **User** is the digital login identity; the **Client** is the legal customer.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from banking_fraud_lab import (
    build_digital_banking_features,
    build_learner_facing_views,
    evaluate_alert_scores,
    extract_feature_importance,
    generate_digital_fraud_scenarios_world,
    recommend_lowest_cost_threshold,
)
from banking_fraud_lab.features import DIGITAL_BANKING_FEATURE_FAMILIES
from banking_fraud_lab.monitoring import (
    check_feature_drift,
    check_monitoring_data_quality,
    check_score_drift,
    decide_alerts,
    inspect_alert_queue,
    record_reviewer_action,
    run_batch_scoring,
    summarise_alert_operations,
)
from banking_fraud_lab.schema import PROTECTED_SCENARIO_ANSWER_KEYS

pd.set_option("display.max_columns", 40)

## Generate Learner-Facing Data And Fit The Baseline

The supervised label comes from generated case outcomes for the digital-banking Detection patterns. Protected answer keys stay separate from the learner-facing views. The modeling frame, the LogisticRegression baseline, and the per-row score are produced exactly as in the v0.7 NovaBank interpretability notebook, so the monitoring flow below starts from the same reproducible score. The frame also carries Banking relationship and User lineage so each monitoring score row traces back to a Client-facing record.

In [ ]:
tables = generate_digital_fraud_scenarios_world(
    seed=42,
    scale="small",
    scenario_prevalence=0.5,
    noisy_outcome_rate=0.3,
)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables

feature_frame = build_digital_banking_features(learner_tables)

model_frame = (
    learner_tables["cases"][["case_id", "alert_id", "transaction_id"]]
    .merge(
        learner_tables["alerts"][
            [
                "alert_id",
                "alert_type",
                "severity",
                "reason",
                "generated_at",
                "alert_status",
                "banking_relationship_id",
                "user_id",
            ]
        ],
        on="alert_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        learner_tables["case_outcomes"][["case_id", "confirmed_fraud", "loss_amount_chf"]],
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(feature_frame, on="transaction_id", how="inner", validate="one_to_one")
)
assert model_frame["confirmed_fraud"].nunique() == 2

feature_output_columns = [
    output_column
    for spec in DIGITAL_BANKING_FEATURE_FAMILIES
    for output_column in spec.output_columns
]
numeric_feature_columns = [
    column
    for column in feature_output_columns
    if pd.api.types.is_numeric_dtype(model_frame[column])
]
assert numeric_feature_columns

preprocess = ColumnTransformer(
    [("numeric", StandardScaler(), numeric_feature_columns)],
    remainder="drop",
)
baseline_model = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "model",
            LogisticRegression(
                random_state=42,
                solver="lbfgs",
                max_iter=1000,
                class_weight="balanced",
            ),
        ),
    ]
)
target = model_frame["confirmed_fraud"].astype(bool)
baseline_model.fit(model_frame[numeric_feature_columns], target)

model_frame = model_frame.assign(
    score=baseline_model.predict_proba(
        model_frame[numeric_feature_columns]
    )[:, 1].round(6)
)
model_frame[
    ["alert_id", "alert_type", "banking_relationship_id", "confirmed_fraud", "score"]
].head()

## Choose The Threshold And Run Batch Scoring

The threshold is sourced from the v0.7 cost-aware recommender (reused, not recomputed). The scored frame carries Banking relationship, transaction, alert, and User lineage so each monitoring score row traces back to a Client-facing record and the digital login identity. `run_batch_scoring` materializes the production `score` and `threshold` tables for the `digital_scam_to_mule` Detection pattern.

In [ ]:
alert_scores = pd.DataFrame(
    {"alert_id": model_frame["alert_id"], "score": model_frame["score"]}
)
threshold_recommendation = recommend_lowest_cost_threshold(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    candidate_thresholds=(0.25, 0.5, 0.75),
    alert_capacities=(5, 10, 25),
    investigation_cost_chf=75.0,
    false_positive_cost_chf=25.0,
    missed_fraud_cost_chf=1_000.0,
)
recommended_threshold = threshold_recommendation["recommended_threshold"]

scored_frame = model_frame[
    ["score", "banking_relationship_id", "transaction_id", "alert_id", "user_id"]
].copy()

batch = run_batch_scoring(
    scored_frame,
    detection_pattern_id="digital_scam_to_mule",
    threshold=recommended_threshold,
    scorer="novabank-logreg",
    score_version="0.8.0",
    evidence_ref="v0.7 recommend_lowest_cost_threshold",
)

threshold_id = str(batch.threshold_rows["threshold_id"].iloc[0])
print(f"batch_id={batch.batch_id} threshold_id={threshold_id}")
batch.score_rows.head()

In [ ]:
batch.threshold_rows

## Decide Alerts Against The Threshold

`decide_alerts` applies the chosen threshold to the score rows, writing `alert_decision` rows (`alert` vs `suppress`) and one `audit_event` row per decision. Each decision carries forward the score, threshold, Banking relationship, and User lineage.

In [ ]:
decisions = decide_alerts(
    batch.score_rows,
    threshold=recommended_threshold,
    threshold_id=threshold_id,
)

decision_split = decisions.alert_decision_rows["decision"].value_counts()
decision_split

In [ ]:
decisions.alert_decision_rows.head()

## Record A Reviewer Action With v0.7 Explanation Evidence

Each alert_decision can carry a reviewer action whose evidence is a v0.7 feature-importance summary. Recording the action emits a second audit event per row, so the score → decision → reviewer-action chain is fully traceable. The evidence here is the top feature driver for `digital_scam_to_mule` from the fitted baseline.

In [ ]:
importance = extract_feature_importance(
    baseline_model,
    numeric_feature_columns,
    feature_frame=model_frame[numeric_feature_columns],
    detection_pattern_id="digital_scam_to_mule",
)
top_row = importance.sort_values("native_importance", ascending=False).iloc[0]
top_feature = str(top_row["feature"])
top_native_importance = float(top_row["native_importance"])

reviewer_actions = record_reviewer_action(
    decisions.alert_decision_rows,
    reviewer="digital-analyst",
    action="confirm",
    rationale="Reviewed against scam-to-mule investigation playbook.",
    evidence={
        "detection_pattern_id": "digital_scam_to_mule",
        "top_feature": top_feature,
        "native_importance": top_native_importance,
    },
)

audit_chain_summary = pd.DataFrame(
    [
        {
            "score_rows": len(batch.score_rows),
            "alert_decision_rows": len(decisions.alert_decision_rows),
            "decision_audit_rows": len(decisions.audit_rows),
            "reviewer_action_rows": len(reviewer_actions.reviewer_action_rows),
            "reviewer_audit_rows": len(reviewer_actions.audit_rows),
        }
    ]
)
audit_chain_summary

In [ ]:
reviewer_actions.reviewer_action_rows.head()

## Inspect The NovaBank Digital Alert Queue

Enriching the alert_decision rows with the foundation Alert lifecycle fields (`generated_at`, `severity`, `alert_status`) and an `institution_name` turns the decision table into an inspectable queue. `inspect_alert_queue` ranks alerts by severity (high > medium > low), then `generated_at`, then `alert_id`, mirroring `sql/examples/04_progressive_alert_queue.sql`, and reports queue age against a fixed `now` (not a wall clock).

In [ ]:
queue_input = decisions.alert_decision_rows.merge(
    learner_tables["alerts"]
    [["alert_id", "generated_at", "severity", "alert_status"]],
    on="alert_id",
    how="inner",
    validate="many_to_one",
)
queue_input["institution_name"] = "NovaBank Digital"

queue = inspect_alert_queue(
    queue_input,
    institution="NovaBank Digital",
    now=pd.Timestamp("2024-01-01"),
)
queue[["alert_queue_rank", "alert_id", "severity", "alert_status", "alert_age_hours"]]

## Summarise Operational Metrics

`summarise_alert_operations` reports alert volume against capacity, the closure distribution by Alert status, and the Detection patterns in scope. Precision and recall are reused from the v0.7 `evaluate_alert_scores` result rather than recomputed, so the operational summary stays consistent with the evaluation layer.

In [ ]:
operations_input = decisions.alert_decision_rows.merge(
    learner_tables["alerts"][["alert_id", "alert_status"]],
    on="alert_id",
    how="inner",
    validate="many_to_one",
)
operations_input["institution_name"] = "NovaBank Digital"

evaluation = evaluate_alert_scores(
    cases=model_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=alert_scores,
    thresholds=(recommended_threshold,),
    alert_capacity=10,
    investigation_cost_chf=75.0,
    false_positive_cost_chf=25.0,
    missed_fraud_cost_chf=1_000.0,
)

operations_summary = summarise_alert_operations(
    operations_input,
    alert_capacity=10,
    evaluation=evaluation,
)
pd.DataFrame([operations_summary])

## Score And Feature Drift

Drift checks compare a reference window against a review window. The score-drift check flags a review when the mean shift exceeds a stated tolerance; the feature-drift check reports per-feature shift so a reviewer can see which input family moved. Here the review window is a controlled upward shift of the reference, simulating drift on synthetic data only.

In [ ]:
reference_scores = batch.score_rows["score"]
review_scores = (reference_scores + 0.2).clip(upper=1.0)

score_drift = check_score_drift(
    reference_scores,
    review_scores,
    tolerance=0.05,
)
pd.DataFrame([score_drift.__dict__])

In [ ]:
reference_features = model_frame[numeric_feature_columns].reset_index(drop=True)
review_features = reference_features.copy()
drift_columns = numeric_feature_columns[:4]
for column in drift_columns:
    review_features[column] = review_features[column] + review_features[column].abs().mean()

feature_drift = check_feature_drift(
    reference_features,
    review_features,
    feature_columns=drift_columns,
)
feature_drift

## Monitoring Data-Quality

`check_monitoring_data_quality` verifies the monitoring input frame has the required columns and wraps the generated-dataset quality report. The result references the v0.7 `data_quality` governance dimension and the `generate_dataset_quality_report` evidence source, so a finding maps onto the monitoring checklist.

In [ ]:
data_quality = check_monitoring_data_quality(
    model_frame,
    required_columns=["score", "banking_relationship_id", "confirmed_fraud"],
)
pd.DataFrame(
    [
        {
            "dimension_id": data_quality.dimension_id,
            "evidence_source": data_quality.evidence_source,
            "row_count": data_quality.row_count,
            "missing_required_columns": list(data_quality.missing_required_columns),
            "passed": data_quality.passed,
        }
    ]
)

## Keep Monitoring Limits Visible

Monitoring is a review prompt, not a verdict. The score, queue, and metrics above surface what to look at — they do not certify the model. Fraud labels are sparse, alert outcomes are operational decisions, and protected answer keys stay separate from learner-facing data. A reviewer still has to read the alert, the explanation, and the case context. **A model should not be judged by headline accuracy.**